# Using `_generate_paper_id`

`_generate_paper_id(nodes_by_source)` returns a deterministic 16-char SHA-256 id for a merged paper.

- Input: `nodes_by_source` — a dict mapping a source name (`wos`, `scopus`, `semantic_scholar`, `acl`, `acm`) to that source's node-attribute dict.
- It picks the **highest-priority identifier** available, scanning sources in the order `wos > scopus > semantic_scholar > acl > acm` and, within a node, identifiers in the order `doi > wos_id > scopus_eid > semantic_scholar_id > acl_id`.
- If no known identifier is found, it falls back to hashing the sorted node ids.

In [ ]:
import sys
sys.path.append("../")  # go up one level from notebooks/ to project root (for using src/)

from src.data_merge import _generate_paper_id

## Example 1 — DOI wins

When a node has a `doi`, it is used first (key `doi:<doi>`), regardless of any other identifiers present.

In [ ]:
nodes_by_source = {
    "wos": {
        "doi": "10.18653/v1/2020.acl-main.1",
        "wos_id": "WOS:000123456700001",
        "title": "Climate change and NLP",
    },
}

paper_id = _generate_paper_id(nodes_by_source)
print("key used: doi:10.18653/v1/2020.acl-main.1")
print("paper_id:", paper_id)

## Example 2 — priority order across sources and identifiers

No DOI anywhere. The function scans sources in priority order, so the `wos` node's `wos_id` is chosen before the `scopus` node's `scopus_eid`.

In [ ]:
nodes_by_source = {
    "scopus": {
        "scopus_eid": "2-s2.0-85099999999",
        "title": "Climate change and NLP",
    },
    "wos": {
        "wos_id": "WOS:000123456700001",
        "title": "Climate change and NLP",
    },
}

paper_id = _generate_paper_id(nodes_by_source)
print("key used: wos:WOS:000123456700001 (wos outranks scopus)")
print("paper_id:", paper_id)

## Example 3 — determinism

The id depends only on the chosen identifier, so it is stable across calls and independent of the insertion order of the sources / other fields.

In [ ]:
a = {
    "wos": {"doi": "10.1000/xyz", "wos_id": "WOS:000111"},
    "scopus": {"scopus_eid": "2-s2.0-85000"},
}
# same identifiers, different insertion order and extra fields
b = {
    "scopus": {"scopus_eid": "2-s2.0-85000"},
    "wos": {"wos_id": "WOS:000111", "doi": "10.1000/xyz", "title": "whatever"},
}

id_a = _generate_paper_id(a)
id_b = _generate_paper_id(b)
print("id_a:", id_a)
print("id_b:", id_b)
print("equal:", id_a == id_b)

## Example 4 — fallback branch

If none of the nodes carry a known identifier, the function falls back to hashing the sorted **node ids** (`_node_id`). Because node ids are unique per source record, two id-less components made of the same set of sources still get different `paper_id`s.

In [ ]:
comp_1 = {
    "acm": {"_node_id": "acm:0001", "title": "Paper A (no ids)"},
    "acl": {"_node_id": "acl:0001", "title": "Paper A (no ids)"},
}
comp_2 = {
    "acm": {"_node_id": "acm:0002", "title": "Paper B (no ids)"},
    "acl": {"_node_id": "acl:0002", "title": "Paper B (no ids)"},
}

id_1 = _generate_paper_id(comp_1)
id_2 = _generate_paper_id(comp_2)
print("id_1:", id_1)
print("id_2:", id_2)
print("distinct despite same sources {acl, acm}:", id_1 != id_2)

## Verification — uniqueness on the merged dataset

`minimal_merge` now asserts `paper_id` uniqueness internally (raising `ValueError` on any duplicate). We can also check the written output directly: every record in `merged_minimal.jsonl` must have a distinct `paper_id`.

In [ ]:
import json
from collections import Counter
from pathlib import Path

merged_path = Path("../data/processed/merged_minimal.jsonl")
ids = []
with open(merged_path) as fh:
    for line in fh:
        ids.append(json.loads(line)["paper_id"])

counts = Counter(ids)
duplicates = {pid: n for pid, n in counts.items() if n > 1}

print("total records :", len(ids))
print("unique ids    :", len(counts))
print("duplicates    :", len(duplicates))
assert not duplicates, f"Duplicate paper_id(s) found: {duplicates}"
print("OK - all paper_id values are unique.")